# cd_A_weakfaith — Runtime Scaling vs d

固定 n，扫描不同 d，拟合 log-time/log-d（幂律）和 log-time/d（指数）两种关系，
找到实践上的最大可行 d。

In [ ]:
import os, sys, time, warnings
import concurrent.futures
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "calm_dataset.py").exists() and (p / "coordinate_descent").exists():
            return p
    raise RuntimeError(f"repo root not found from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from calm_dataset import CalmDataset
print(f"REPO_ROOT: {REPO_ROOT}")


In [ ]:
ALGO_NAME = "cd_A_weakfaith"
TAG       = "cd_A_weakfaith"

from coordinate_descent.cd_A_weakfaith import dag_coordinate_descent_l0_weakfaith
print("cd_A_weakfaith: OK")


In [ ]:
D_LIST      = [10, 20, 50, 100, 200, 300, 500, 750, 1000]
N_SAMPLES   = 1000
TRIALS      = 3
DEGREE      = 2.0
GRAPH_TYPE  = "ER"
SEED_BASE   = 42
TIMEOUT_SEC = 3600
os.makedirs(str(REPO_ROOT / "experiments" / "results"), exist_ok=True)

In [ ]:
def run_with_timeout(fn, timeout):
    """Returns (elapsed_sec, error_str_or_None)."""
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(fn)
    t0 = time.perf_counter()
    try:
        fut.result(timeout=timeout)
        elapsed = time.perf_counter() - t0
        ex.shutdown(wait=False)
        return elapsed, None
    except concurrent.futures.TimeoutError:
        ex.shutdown(wait=False)
        return float(timeout), "TIMEOUT"
    except Exception as e:
        elapsed = time.perf_counter() - t0
        ex.shutdown(wait=False)
        return elapsed, str(e)


def make_data(d, seed):
    ds = CalmDataset(
        n=N_SAMPLES, d=d, graph_type=GRAPH_TYPE,
        degree=DEGREE, sem_type="gauss",
        noise_ratio=4.0, seed=int(seed),
    )
    return ds.X

In [ ]:
def run_algo(X: np.ndarray):
    n = X.shape[0]
    S = X.T @ X / n
    dag_coordinate_descent_l0_weakfaith(
        S=S, T=1000000, seed=0, threshold=0.05, lambda_l0=0.2,
        faithfulness_tau=0.05, screening=["corr", "pcorr"], combine="union",
    )


## 主循环

In [ ]:
records = []
stopped_at_d = None
rng = np.random.default_rng(SEED_BASE)
seeds = rng.integers(0, 10**9, size=(len(D_LIST), TRIALS))

for di, d in enumerate(D_LIST):
    if stopped_at_d is not None:
        break
    trial_times = []
    for t in range(TRIALS):
        X = make_data(d, int(seeds[di, t]))
        elapsed, err = run_with_timeout(lambda X=X: run_algo(X), TIMEOUT_SEC)
        if err == "TIMEOUT":
            print(f"d={d:4d}  trial={t+1}  TIMEOUT (>{TIMEOUT_SEC}s)")
            stopped_at_d = d
            break
        elif err:
            print(f"d={d:4d}  trial={t+1}  ERROR: {err}")
        else:
            trial_times.append(elapsed)
            print(f"d={d:4d}  trial={t+1}  {elapsed:.3f}s")
    if trial_times:
        records.append({"d": d, "mean_sec": np.mean(trial_times),
                        "std_sec": np.std(trial_times), "n_trials": len(trial_times)})

df = pd.DataFrame(records)
print()
print(df.to_string(index=False))


## 可视化

In [ ]:
if df.empty or len(df) < 3:
    print("Not enough data points to plot.")
else:
    d_arr = df["d"].values.astype(float)
    t_arr = df["mean_sec"].values
    s_arr = df["std_sec"].values

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"{ALGO_NAME} — runtime scaling  (n={N_SAMPLES}, ER degree={DEGREE})", fontsize=13)

    # ── log-log: power law ───────────────────────────────────────────────────
    ax = axes[0]
    ax.errorbar(d_arr, t_arr, yerr=s_arr, fmt="o-", capsize=4, label="measured")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("d  (log scale)"); ax.set_ylabel("time (s,  log scale)")
    ax.set_title("log-log  →  power law  t ∝ dᵏ ?")
    ax.grid(True, which="both", ls="--", alpha=0.4)
    sl, ic, r, *_ = stats.linregress(np.log(d_arr), np.log(t_arr))
    xf = np.logspace(np.log10(d_arr.min()), np.log10(d_arr.max()), 100)
    ax.plot(xf, np.exp(ic) * xf**sl, "r--",
            label=f"fit: t ∝ d^{sl:.2f}   R²={r**2:.3f}")
    ax.legend()

    # ── log-linear: exponential ──────────────────────────────────────────────
    ax = axes[1]
    ax.errorbar(d_arr, t_arr, yerr=s_arr, fmt="o-", capsize=4, label="measured")
    ax.set_yscale("log")
    ax.set_xlabel("d"); ax.set_ylabel("time (s,  log scale)")
    ax.set_title("log-linear  →  exponential  t ∝ exp(k·d) ?")
    ax.grid(True, which="both", ls="--", alpha=0.4)
    sl2, ic2, r2, *_ = stats.linregress(d_arr, np.log(t_arr))
    xf2 = np.linspace(d_arr.min(), d_arr.max(), 100)
    ax.plot(xf2, np.exp(ic2 + sl2 * xf2), "r--",
            label=f"fit: t ∝ exp({sl2:.4f}·d)   R²={r2**2:.3f}")
    ax.legend()

    plt.tight_layout()
    out_png = REPO_ROOT / "experiments" / "results" / f"{TAG}_scaling.png"
    plt.savefig(out_png, dpi=120)
    plt.show()
    print(f"figure saved → {out_png}")


## 结论

In [ ]:
if len(df) >= 3:
    d_arr = df["d"].values.astype(float)
    t_arr = df["mean_sec"].values
    _, _, r_ll, *_ = stats.linregress(np.log(d_arr), np.log(t_arr))
    sl_ll, _, _, *_ = stats.linregress(np.log(d_arr), np.log(t_arr))
    sl_le, _, r_le, *_ = stats.linregress(d_arr, np.log(t_arr))
    # recompute properly
    sl_ll, ic_ll, r_ll, *_ = stats.linregress(np.log(d_arr), np.log(t_arr))
    sl_le, ic_le, r_le, *_ = stats.linregress(d_arr, np.log(t_arr))

    print("=== Scaling law summary ===")
    print(f"  log-log    R² = {r_ll**2:.4f}  →  t ∝ d^{sl_ll:.2f}  (power law / polynomial)")
    print(f"  log-linear R² = {r_le**2:.4f}  →  t ∝ exp({sl_le:.4f}·d)  (exponential)")
    if r_ll**2 > r_le**2:
        print(f"  Better fit: POWER LAW  (exponent ≈ {sl_ll:.2f})")
    else:
        print(f"  Better fit: EXPONENTIAL  (rate ≈ {sl_le:.4f} per node)")
    if stopped_at_d:
        print(f"  Practical max d  <  {stopped_at_d}  (timeout = {TIMEOUT_SEC}s, n = {N_SAMPLES})")
    else:
        print(f"  All d tested within timeout — practical max d ≥ {D_LIST[-1]}  (n = {N_SAMPLES})")
else:
    print("Not enough data for scaling analysis.")
